# Notebook 01: Data Download & Preparation (VIRVS Benchmark)

Welcome to the practical workshop on **Virus Infection Reporter Virtual Staining (VIRVS)**!

### Overview
In this notebook, we explore the dataset structure introduced in the VIRVS benchmark ([Wyrzykowska et al., 2024](https://github.com/casus/virvs)). The benchmark maps non-destructive label-free **Brightfield** micrographs ($y$) to continuous **Fluorescence** infection reporter signals ($x$).

**Key Learning Objectives**:
1. Set up the environment (with 1-click Google Colab support).
2. Understand dataset pair structure for virtual staining.
3. Generate/load paired micrographs.
4. Implement PyTorch `Dataset` & `DataLoader` pipelines.

In [ ]:
# ==========================================================
# 1. Google Colab Setup & Environment Setup
# ==========================================================
import sys
import os

if 'google.colab' in sys.modules:
    print("[+] Google Colab detected! Installing dependencies and setting up paths...")
    !git clone https://github.com/casus/GenAI_BIA_Course.git /content/GenAI_BIA_Course
    %cd /content/GenAI_BIA_Course/practical
    !pip install -r ../requirements.txt -q
    sys.path.append(os.path.abspath("."))
else:
    print("[+] Local execution detected.")
    sys.path.append(os.path.abspath("."))

## 2. Generate Synthetic VIRVS Data Pair
To make the workshop runnable instantly without downloading multi-gigabyte raw datasets, we use `src.generate_mock_virvs_data` to construct high-fidelity paired micrographs.

In [ ]:
import matplotlib.pyplot as plt
from src.generate_mock_virvs_data import create_dataset_directory, generate_virvs_pair

data_dir = "./data/mock_virvs"
create_dataset_directory(data_dir, num_train=30, num_val=10, image_size=(256, 256))

# Visualize a single synthetic sample pair
bf, fluo, masks = generate_virvs_pair(image_size=(256, 256), seed=42)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(bf, cmap="gray")
axes[0].set_title("Input: Brightfield (y)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(fluo, cmap="magma")
axes[1].set_title("Target: Virus Reporter (x)", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(masks.sum(axis=0), cmap="viridis")
axes[2].set_title("Cell Masks (Ground Truth)", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 3. PyTorch Data Pipeline
We load the generated dataset into our PyTorch `VIRVSDataset` class and inspect sample tensor batches.

In [ ]:
from torch.utils.data import DataLoader
from src.data import VIRVSDataset

# Instantiate PyTorch training & validation datasets
train_dataset = VIRVSDataset(root_dir=data_dir, split="train", normalize_range=(-1.0, 1.0))
val_dataset = VIRVSDataset(root_dir=data_dir, split="val", normalize_range=(-1.0, 1.0))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

batch = next(iter(train_loader))
print("Brightfield tensor shape:", batch["brightfield"].shape)  # [B, 1, H, W]
print("Fluorescence tensor shape:", batch["fluorescence"].shape) # [B, 1, H, W]
print("Min/Max range:", batch["brightfield"].min().item(), batch["brightfield"].max().item())